# 🤖 Ollama LLM Backend — Google Colab

Ce notebook fait tourner Ollama avec un modèle LLM sur le GPU T4 gratuit de Colab.
Il expose l'API via ngrok et se régénère toutes les 24h.

**Instructions:**
1. Runtime > Change runtime type > GPU (T4)
2. Exécuter toutes les cellules dans l'ordre
3. Copier l'URL ngrok affichée dans le `.env` du bot ou utiliser le mode dynamique

In [ ]:
# Cell 1: Install Ollama + ngrok
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok requests

In [ ]:
# Cell 2: Configuration
MODEL_NAME = 'qwen2.5:7b'  # 7B fits on T4 (16GB VRAM)
NGROK_AUTHTOKEN = ''  # <-- PASTE YOUR NGROK AUTHTOKEN HERE
REGENERATE_HOURS = 24  # Session regeneration interval

# Webhook URL to notify the bot of URL changes (optional)
# Set this to your bot's control server URL
BOT_WEBHOOK_URL = ''  # e.g. https://your-vps.com/webhook/colab-url

In [ ]:
# Cell 3: Start Ollama server in background
import subprocess, os, time, signal

# Kill any existing Ollama
os.system('pkill ollama')
time.sleep(2)

# Start Ollama server
ollama_proc = subprocess.Popen(['ollama', 'serve'], 
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)
print('✅ Ollama server started')

# Pull model
print(f'⏳ Pulling model {MODEL_NAME}...')
os.system(f'ollama pull {MODEL_NAME}')
print(f'✅ Model {MODEL_NAME} ready')

In [ ]:
# Cell 4: Start ngrok tunnel
from pyngrok import ngrok, conf
import requests, json

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

# Kill existing tunnels
ngrok.kill()
time.sleep(2)

# Start tunnel on port 11434 (Ollama default)
tunnel = ngrok.connect(11434, 'http')
OLLAMA_URL = tunnel.public_url
print(f'🌐 Ollama public URL: {OLLAMA_URL}')
print(f'   → Set LOCAL_LLM_URL={OLLAMA_URL}/v1 in your bot .env')

# Notify bot webhook if configured
if BOT_WEBHOOK_URL:
    try:
        resp = requests.post(BOT_WEBHOOK_URL, json={'url': OLLAMA_URL, 'model': MODEL_NAME})
        print(f'📡 Bot notified: {resp.status_code}')
    except Exception as e:
        print(f'⚠️ Webhook failed: {e}')

In [ ]:
# Cell 5: Keep-alive + auto-regeneration loop
# This cell runs until Colab session expires, then you re-run the notebook

import time, requests, json, subprocess
from datetime import datetime, timedelta

start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 Session started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")} ({REGENERATE_HOURS}h)')
print(f'📊 Monitoring loop started (checks every 60s)...')
print(f'   Press Ctrl+C to stop manually')

health_ok_count = 0
health_fail_count = 0

try:
    while True:
        now = datetime.now()
        elapsed = now - start_time
        remaining = regenerate_at - now
        
        # Health check Ollama
        try:
            resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=10)
            if resp.status_code == 200:
                health_ok_count += 1
                status = '✅ healthy'
            else:
                health_fail_count += 1
                status = f'⚠️ status {resp.status_code}'
        except Exception as e:
            health_fail_count += 1
            status = f'❌ {str(e)[:50]}'
        
        # Print status every 5 minutes
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            print(f'[{now.strftime("%H:%M:%S"}] uptime={int(elapsed.total_seconds()/60)}min '
                  f'remaining={int(remaining.total_seconds()/60)}min '
                  f'ok={health_ok_count} fail={health_fail_count} {status}')
        
        # Check if it's time to regenerate
        if now >= regenerate_at:
            print(f'🔄 Regeneration triggered at {now.strftime("%H:%M:%S")}')
            print('   Restarting ngrok tunnel...')
            
            # Kill old tunnel
            ngrok.kill()
            time.sleep(3)
            
            # Start new tunnel
            new_tunnel = ngrok.connect(11434, 'http')
            OLLAMA_URL = new_tunnel.public_url
            print(f'   New URL: {OLLAMA_URL}')
            
            # Notify bot
            if BOT_WEBHOOK_URL:
                try:
                    requests.post(BOT_WEBHOOK_URL, json={'url': OLLAMA_URL, 'model': MODEL_NAME})
                    print('   📡 Bot notified of new URL')
                except Exception as e:
                    print(f'   ⚠️ Webhook failed: {e}')
            
            # Reset timer
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
            print(f'   Next regeneration at {regenerate_at.strftime("%H:%M:%S")}')
        
        time.sleep(60)
        
except KeyboardInterrupt:
    print('\n⏹️ Stopped by user')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Session stats: ok={health_ok_count} fail={health_fail_count}')

In [ ]:
# Cell 6 (optional): Test the LLM
import requests, json

test_url = f'{OLLAMA_URL}/v1/chat/completions'
payload = {
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Bonjour, réponds en une phrase.'}],
    'max_tokens': 50,
    'stream': False
}

resp = requests.post(test_url, json=payload, timeout=60)
print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    print(f'Response: {data["choices"][0]["message"]["content"]}')
else:
    print(f'Error: {resp.text[:200]}')